# D170 — Analytical SQL 2: Intern Hands-on Exercises

This notebook is a hands-on lab using the Olist tables loaded in D16. Each exercise gives a business problem, requirements, and an expected result shape. Write and run your SQL in the empty cell below each problem.

## Learning goals

- combine related tables with `INNER JOIN`;
- filter detail rows with `WHERE`;
- summarize data with `COUNT`, `MIN`, `MAX`, `SUM`, and `AVG`;
- create business-level groups with `GROUP BY`;
- filter summarized groups with `HAVING`; and
- sort and limit results with `ORDER BY` and `LIMIT`.

Do not use CTEs, window functions, or subqueries in this lab. Every problem can be solved with simple joins and aggregate queries.


## 1. Connect to MySQL

The defaults match the D16 classroom database. Environment variables can override them when needed.


In [ ]:
import os
import mysql.connector
from mysql.connector import Error

connection = mysql.connector.connect(
    host=os.environ.get("MYSQL_HOSTNAME", "127.0.0.1"),
    port=int(os.environ.get("MYSQL_PORT", "3306")),
    user=os.environ.get("MYSQL_USERNAME", "root"),
    password=os.environ.get("MYSQL_PASSWORD", "root"),
    database=os.environ.get("MYSQL_DATABASE", "olist_import_lab"),
)

print("Connected:", connection.is_connected())
print("MySQL version:", connection.server_info)


## 2. Query helpers

`execute_sql` sends a query to MySQL. `print_rows` displays the returned records as an aligned table. A maximum of 25 rows is printed by default.


In [ ]:
def print_rows(columns, rows):
    if not rows:
        print("No rows returned.")
        return

    text_rows = [["NULL" if value is None else str(value) for value in row]
                 for row in rows]
    widths = [len(str(column)) for column in columns]
    for row in text_rows:
        widths = [max(width, len(value)) for width, value in zip(widths, row)]

    print(" | ".join(str(column).ljust(width)
                     for column, width in zip(columns, widths)))
    print("-+-".join("-" * width for width in widths))
    for row in text_rows:
        print(" | ".join(value.ljust(width)
                         for value, width in zip(row, widths)))


def execute_sql(sql, params=None, max_rows=25):
    cursor = connection.cursor()
    try:
        cursor.execute(sql, params or ())
        if not cursor.with_rows:
            connection.commit()
            print(f"Statement completed. Affected rows: {cursor.rowcount:,}")
            return cursor.rowcount

        columns = [item[0] for item in cursor.description]
        rows = cursor.fetchmany(max_rows + 1)
        more_rows = len(rows) > max_rows
        visible_rows = rows[:max_rows]
        print_rows(columns, visible_rows)
        if more_rows:
            print(f"... showing the first {max_rows} rows")
        return visible_rows
    except Error:
        connection.rollback()
        raise
    finally:
        cursor.close()


## 3. Tables used in this lab

| Table | Important columns | Meaning |
|---|---|---|
| `olist_customers` | `customer_id`, `customer_unique_id`, `customer_city`, `customer_state` | Customer and location |
| `olist_orders` | `order_id`, `customer_id`, `order_status`, `order_purchase_timestamp` | One row per order |
| `olist_order_items` | `order_id`, `product_id`, `seller_id`, `price`, `freight_value` | Items sold in an order |
| `olist_order_payments` | `order_id`, `payment_type`, `payment_installments`, `payment_value` | Payments made for orders |
| `olist_order_reviews` | `order_id`, `review_score` | Customer review scores |
| `olist_products` | `product_id`, `product_category_name` | Product details |
| `product_category_translation` | `product_category_name`, `product_category_name_english` | English category names |
| `olist_sellers` | `seller_id`, `seller_city`, `seller_state` | Seller and location |

Useful join path:

`customers → orders → order_items → products / sellers`

An order can have several item rows and several payment rows. Avoid joining items and payments together unless the problem specifically requires it, because that can multiply rows and inflate totals.


In [ ]:
execute_sql("""
SELECT 'customers' AS table_name, COUNT(*) AS row_count FROM olist_customers
UNION ALL
SELECT 'orders', COUNT(*) FROM olist_orders
UNION ALL
SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL
SELECT 'payments', COUNT(*) FROM olist_order_payments
UNION ALL
SELECT 'reviews', COUNT(*) FROM olist_order_reviews
""")


# Exercises

Use clear aliases for tables and calculated columns. Round monetary averages and totals to two decimal places. Add a stable tie-breaker to `ORDER BY` when practical.


## Exercise 1: Customer count by state

The customer-success team wants to understand the geographic distribution of customer records.

**Task:** Using `olist_customers`, return each customer state and its number of customer records. Show the largest count first.

**Must use:** `COUNT`, `GROUP BY`, `ORDER BY`.

**Expected result:** 27 rows (one per state), with columns `customer_state` and `customer_count`. The first row should be the state with the most customers.


## Exercise 2: Top 10 spending customers

The retention team wants the buyers with the greatest total payments.

**Task:** Join customers, orders, and payments. Group by `customer_unique_id`, then return the 10 buyers with the largest sum of `payment_value`. Also show how many distinct orders each buyer placed.

**Must use:** two `INNER JOIN`s, `COUNT(DISTINCT ...)`, `SUM`, `GROUP BY`, `ORDER BY`, `LIMIT`.

**Expected result:** 10 rows with columns `customer_unique_id`, `order_count`, and `total_spent`; sorted from highest to lowest `total_spent`. Round the total to two decimals.


## Exercise 3: Payment summary by type

Finance wants to compare the payment methods used by customers.

**Task:** Using `olist_order_payments`, exclude `not_defined`. For every remaining payment type, calculate the number of payment rows, minimum payment, maximum payment, average payment, and total payment value.

**Must use:** `WHERE`, `COUNT`, `MIN`, `MAX`, `AVG`, `SUM`, `GROUP BY`.

**Expected result:** One row per valid payment type with columns `payment_type`, `payment_count`, `minimum_payment`, `maximum_payment`, `average_payment`, and `total_payment`. Round monetary values to two decimals and order by total descending.


## Exercise 4: Large delivered orders

Operations wants to inspect delivered orders that contain several items.

**Task:** Join orders and order items. Keep only orders whose status is `delivered`. Calculate item count, total item price, total freight, and the combined order value for each order. Return only orders containing at least 5 items.

**Must use:** `INNER JOIN`, `WHERE`, `COUNT`, `SUM`, `GROUP BY`, `HAVING`.

**Expected result:** Columns `order_id`, `item_count`, `item_total`, `freight_total`, and `order_total`. Sort by item count descending, then order total descending. Show the first 10 rows.


## Exercise 5: High-volume sellers by state

The marketplace team wants states containing sellers with meaningful sales activity.

**Task:** Join sellers and order items. Summarize each seller state with its distinct seller count, items sold, and total item revenue. Keep only states with at least 100 sellers and total item revenue greater than 1,000,000.

**Must use:** `INNER JOIN`, `COUNT(DISTINCT ...)`, `COUNT`, `SUM`, `GROUP BY`, `HAVING`.

**Expected result:** Columns `seller_state`, `seller_count`, `items_sold`, and `item_revenue`; ordered by item revenue descending. All returned rows must satisfy both HAVING conditions.


## Exercise 6: Best-selling product categories

Merchandising wants to know which English product categories generate the most item revenue.

**Task:** Join order items to products and then to category translation. Exclude rows where the English category name is null. For each English category, calculate distinct orders, items sold, average item price, and total item revenue. Return the top 10 by revenue.

**Must use:** two `INNER JOIN`s, `WHERE`, `COUNT(DISTINCT ...)`, `COUNT`, `AVG`, `SUM`, `GROUP BY`, `LIMIT`.

**Expected result:** 10 rows with columns `category`, `order_count`, `items_sold`, `average_item_price`, and `item_revenue`; highest revenue first.


## Exercise 7: Review score by order status

Customer experience wants to compare satisfaction across order statuses.

**Task:** Join orders and reviews. For each order status, calculate the number of reviews, minimum score, maximum score, and average score. Include only statuses having at least 10 reviews.

**Must use:** `INNER JOIN`, `COUNT`, `MIN`, `MAX`, `AVG`, `GROUP BY`, `HAVING`.

**Expected result:** Columns `order_status`, `review_count`, `minimum_score`, `maximum_score`, and `average_score`. Average score should be rounded to two decimals and sorted highest first.


## Exercise 8: Freight analysis by customer state

Logistics wants to compare freight charges across customer states for delivered orders.

**Task:** Join customers, orders, and order items. Keep only delivered orders. For each customer state, calculate item count, minimum freight, maximum freight, average freight, and total freight.

**Must use:** two `INNER JOIN`s, `WHERE`, `COUNT`, `MIN`, `MAX`, `AVG`, `SUM`, `GROUP BY`.

**Expected result:** 27 or fewer state rows with columns `customer_state`, `item_count`, `minimum_freight`, `maximum_freight`, `average_freight`, and `total_freight`; ordered by total freight descending.


## Exercise 9: Cities with strong delivered-order volume

The regional team wants cities with a substantial number of delivered orders.

**Task:** Join customers and orders, filter to delivered orders, and group by customer state and city. Keep only city/state groups with at least 500 distinct orders.

**Must use:** `INNER JOIN`, `WHERE`, `COUNT(DISTINCT ...)`, `GROUP BY`, `HAVING`.

**Expected result:** Columns `customer_state`, `customer_city`, and `delivered_orders`. Every row must contain at least 500 orders. Sort by order count descending, then state and city.


## Exercise 10: Premium-item categories

The product team defines a premium item as one priced at 500 or more. They want to compare categories within this segment.

**Task:** Join order items, products, and category translation. Keep item rows with `price >= 500` and a non-null English category. For each category, calculate premium item count, minimum price, maximum price, average price, and total revenue. Keep categories having at least 20 premium items.

**Must use:** two `INNER JOIN`s, `WHERE`, `MIN`, `MAX`, `AVG`, `SUM`, `GROUP BY`, `HAVING`.

**Expected result:** Columns `category`, `premium_item_count`, `minimum_price`, `maximum_price`, `average_price`, and `total_revenue`; ordered by premium item count descending.


## Exercise 11: Credit-card installment performance

Finance wants to see how order payment values vary by credit-card installment count.

**Task:** Use payments, keep only `credit_card` payments with installment counts from 1 through 12, and group by installment count. Calculate payment count, minimum value, maximum value, average value, and total value. Keep installment groups having at least 100 payments.

**Must use:** `WHERE`, `BETWEEN`, `COUNT`, `MIN`, `MAX`, `AVG`, `SUM`, `GROUP BY`, `HAVING`.

**Expected result:** Columns `payment_installments`, `payment_count`, `minimum_payment`, `maximum_payment`, `average_payment`, and `total_payment`; sorted by installment count ascending.


## Exercise 12: Sales where seller and customer share a state

The marketplace team wants to measure local, same-state commerce.

**Task:** Join customers, orders, order items, and sellers. Keep rows where the seller state equals the customer state. Group by that shared state and calculate distinct orders, distinct sellers, items sold, average item price, and total item revenue. Keep states having at least 500 distinct orders.

**Must use:** three `INNER JOIN`s, `WHERE` comparing two columns, `COUNT(DISTINCT ...)`, `COUNT`, `AVG`, `SUM`, `GROUP BY`, `HAVING`.

**Expected result:** Columns `state`, `order_count`, `seller_count`, `items_sold`, `average_item_price`, and `item_revenue`; ordered by item revenue descending. Every row represents only same-state sales.


## Optional review checklist

Before considering an answer complete, check that:

- every joined table has an explicit `ON` condition;
- `WHERE` filters individual source rows and `HAVING` filters groups;
- every selected non-aggregate column appears in `GROUP BY`;
- currency outputs are rounded to two decimal places;
- aliases explain the business meaning of each result column; and
- the output ordering matches the problem statement.


## Close the connection

Run this cell after finishing the exercises.


In [ ]:
if connection.is_connected():
    connection.close()
print("MySQL connection closed.")
